# Phase 2: GeoJSON streaming export

This notebook implements the Phase 2 processing pipeline. Here, we add synthetic lat/lon to the telemetry stream, which allows our simulated agents to have coordinate trajectories. The stream is synchronically processed via Spark Structured Streaming. This exports each micro-batch into a `GeoJSON` `FeatureCollection` format for geospatial analysis. 

*Note*: The coordinates are simulated programmatically for pipeline demonstration purposes and do not represent actual GPS observations.

In [1]:
import os
import sys
import logging
from pathlib import Path

# Climb up from the notebook's folder to find the true project workspace root
notebook_dir = Path(os.getcwd())
PROJECT_ROOT = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bootstrapping import setup_winutils
from pyspark.sql import SparkSession

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configure Windows-specific local Spark settings dynamically
if os.name == 'nt':
    setup_winutils(PROJECT_ROOT)
    os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark_builder = (
    SparkSession.builder
    .appName('Geospatial-Streaming-Export')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '4')
)

if os.name == 'nt':
    spark_builder = (
        spark_builder
        .config('spark.driver.host', '127.0.0.1')
        .config('spark.pyspark.python', sys.executable)
        .config('spark.pyspark.driver.python', sys.executable)
    )

spark = spark_builder.getOrCreate()
logging.info(f'Project root: {PROJECT_ROOT}')


2026-07-07 23:29:16,665 - INFO - Hadoop environment path configuration active: HADOOP_HOME=c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils
2026-07-07 23:29:22,175 - INFO - Project root: c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling


## 2.1 Spatial Trajectory Generation and Coordinate Projection

As out sensor data timesries has a dimension of 1, for a 2D geographical space the pipeline attaches the synthetically generated latitude ($\phi$) and longitude ($\lambda$) to each agent.

#### 1. Velocity Models
An agent's speed $v$ is determined by its classified activity state:
$$v = \begin{cases} 1.4\text{ m/s} & \text{if Activity} = \text{"walk"} \\ 4.5\text{ m/s} & \text{if Activity} = \text{"bike"} \\ 0.0\text{ m/s} & \text{otherwise} \end{cases}$$

#### 2. Angular Heading Direction
The walking or cycling heading direction $\theta$ (in radians) is determined by the agent's identifier modulo 16, dividing the compass into 16 discrete angles:
$$\theta = (\text{Agent\_ID} \pmod{16}) \times \frac{2\pi}{16}$$

#### 3. Spatial Displacement Mapping
Given the elapsed time $t$ since the base timestamp, the displacement in meters along the East ($\Delta x$) and North ($\Delta y$) axes is calculated as:
$$\Delta x = v \cdot t \cdot \cos(\theta)$$
$$\Delta y = v \cdot t \cdot \sin(\theta)$$

Using a local projection centered on Vienna ($\phi_{\text{start}} = 48.20849^\circ\text{ N}$, $\lambda_{\text{start}} = 16.37208^\circ\text{ E}$), we convert these displacements to degree coordinates:
$$\phi = \phi_{\text{start}} + \frac{\Delta y}{M_{\text{lat}}}$$
$$\lambda = \lambda_{\text{start}} + \frac{\Delta x}{M_{\text{lon}}}$$
where the meters-to-degrees conversion factors are defined by:
$$M_{\text{lat}} = 111,320.0 \text{ m/degree}$$
$$M_{\text{lon}} = 111,320.0 \cdot \cos(\phi_{\text{start}}) \text{ m/degree}$$

In [2]:
from src.geospatial_streaming import run_geospatial_streaming_simulation

output_dir = run_geospatial_streaming_simulation(spark, PROJECT_ROOT)
geojson_files = sorted(output_dir.glob('*.geojson'))
print(f'Created {len(geojson_files)} GeoJSON micro-batch files in {output_dir}')
geojson_files[:3]

2026-07-07 23:29:40,552 - INFO - Layer already cached: vienna_districts.geojson
2026-07-07 23:29:40,556 - INFO - Layer already cached: vienna_pedestrian_zones.geojson
2026-07-07 23:29:40,558 - INFO - Layer already cached: vienna_bike_paths.geojson
2026-07-07 23:29:43,936 - INFO - Loading cached street graph: vienna_walk_network.graphml
2026-07-07 23:31:03,175 - INFO - Callback Server Starting
2026-07-07 23:31:03,191 - INFO - Socket listening on ('127.0.0.1', 60329)
2026-07-07 23:31:06,481 - INFO - Python Server ready to receive messages
2026-07-07 23:31:06,507 - INFO - Received command c on object id p0
2026-07-07 23:31:16,505 - INFO - GeoJSON micro-batch written to c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output\telemetry_batch_00000.geojson
2026-07-07 23:31:18,196 - INFO - Received command c on object id p0
2026-07-07 23:31:24,874 - INFO - GeoJSON micro-batch written to c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturatio

Created 10 GeoJSON micro-batch files in c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output


[WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00000.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00001.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00002.geojson')]

In [3]:
import json

with geojson_files[0].open('r', encoding='utf-8') as file_handle:
    first_batch = json.load(file_handle)

print(first_batch['type'])
print(f"Features in first batch: {len(first_batch['features'])}")
first_batch['features'][0]

FeatureCollection
Features in first batch: 50002


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [16.3742637, 48.1840581]},
 'properties': {'Agent_ID': 0,
  'Timestamp': 1700000000180,
  'Activity': 'walk',
  'ax': -3.512752616073503,
  'ay': -8.776084052302739,
  'az': 19.821129797751485,
  'gx': -1.2284121539662356,
  'gy': 0.17962828206343617,
  'gz': 1.340964162088716,
  'coordinate_source': 'synthetic'}}

## 2.2 Interpretation and limitation

The generated GeoJSON files demonstrate how a Spark Structured Streaming pipeline can attach a geographical representation to synthetic telemetry. The coordinates are generated from explicit speed and direction assumptions; they are not observed GPS positions and are not snapped to official Vienna transport infrastructure. Consequently, the output is suitable for demonstrating data processing and visualisation, but not for drawing conclusions about actual Vienna mobility or congestion.

In [4]:
spark.stop()